# HiData basics: genotypes

cellink works on `HiData` objects from the [HiData](https://hidata.readthedocs.io) package. A `HiData`
links single-cell data `C` to data on the groups the cells come from, `G`, and optionally to subgroups in between, `S`.
The HiData [getting started](https://hidata.readthedocs.io/en/latest/tutorials/getting_started.html) tutorial
shows how such an object is built, selected, summarized and saved.

In cellink, the group level `G` holds the genotypes: an `AnnData` with one row per donor and one column per variant.
This tutorial covers what cellink adds for them: reading genotype files into `G`, the variant annotations cellink
expects, and storing genotypes in compact Zarr stores that are read lazily. It runs no analysis.

In [1]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from anndata import AnnData

import cellink as cl

tmp_dir = Path(tempfile.mkdtemp())

site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## Reading genotype files

`cellink.io` reads genotype files into an `AnnData` with donors as rows and variants as columns, the form `G` takes.

### From PGEN, via `stream_pgen_to_zarr`

For most workflows this is the one to reach for: a single function call directly on the compact PLINK2 format most
GWAS/biobank pipelines already produce, or can export to via `plink2 --make-pgen`. It streams the file in fixed-size
blocks so memory use stays bounded regardless of file size, and writes the genotypes to a Zarr store as 2-bit calls,
four per byte. `read_pgen_zarr` loads the result back with a Dask-backed `X`. The `cellink-pgen` command-line tool
({doc}`../api/cli`) wraps the same conversion for use outside Python.

It does have a few tuning knobs, but the defaults are reasonable for typical use and you usually don't need to touch
them:

- `chunk_samples=4096` / `chunk_variants=2048`: the output Zarr store's chunk shape. Only worth changing for an unusual
  access pattern (e.g. mostly per-variant lookups on a very large cohort) or a dataset far smaller/larger than these
  defaults assume.
- `memory_limit_gb=10.0`: caps how many variants get buffered per read block. Lower it on memory-constrained machines,
  raise it if you have RAM to spare and want faster conversion.
- `genotype_encoding="2bit"`: stores PLINK's hard calls as 2-bit codes. `"int8"` stores one byte per call instead.

In [2]:
from cellink.io import read_pgen_zarr, stream_pgen_to_zarr

pgen_out = tmp_dir / "pgen_demo.zarr"
stream_pgen_to_zarr(
    "../../tests/data/simulated_genotype_calls.pgen",
    str(pgen_out),
    # small values here just because this demo file only has 100 samples / 1000 variants;
    # leave chunk_samples/chunk_variants/memory_limit_gb at their defaults for real data
    chunk_samples=50,
    chunk_variants=200,
    memory_limit_gb=2.0,
)
gdata = read_pgen_zarr(str(pgen_out))
gdata

[2026-11-23 23:00:07,722] INFO:cellink.io._pgen:   simulated_genotype_calls.pgen: 100 samples x 1,000 variants (using 1,000)


[2026-11-23 23:00:07,723] INFO:cellink.io._pgen: Total: 100 samples × 1,000 variants (dense)


[2026-11-23 23:00:07,723] INFO:cellink.io._pgen: Loading sample metadata from ../../tests/data/simulated_genotype_calls.psam ...


[2026-11-23 23:00:07,782] INFO:cellink.io._pgen: Creating dense X (100 × 1,000) chunks=(50, 200) ...


[2026-11-23 23:00:07,802] INFO:cellink.io._pgen: Reading (dense) simulated_genotype_calls.pgen ...


  simulated_genotype_calls:   0%|          | 0/1000 [00:00<?, ?var/s]

[2026-11-23 23:00:08,258] INFO:cellink.io._pgen: Writing obs / var / metadata ...


[2026-11-23 23:00:08,849] INFO:cellink.io._pgen: ✓ Done → /tmp/tmpwn4lf0ga/pgen_demo.zarr


site-packages/zarr/api/asynchronous.py:260: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


AnnData object with n_obs × n_vars = 100 × 1000
    obs: 'iid', 'sex'
    var: 'chrom', 'pos', 'snp_id', 'a0', 'a1'
    uns: 'n_missing'
    layers: None (.X)

### Variant annotations

Every variant is described in `var` by its chromosome `chrom`, its position `pos` and its two alleles `a0` and `a1`.
The genotypes count copies of `a1`, the alternative allele of the PGEN file. These four columns are what cellink's
tools rely on, for example to export genotypes to PLINK or to find the variants near a gene, and
{func}`cellink.validate` checks them. Further columns, such as the variant identifier `snp_id`, are kept as they are.

In [3]:
gdata.var.head()

,chrom,pos,snp_id,a0,a1
sid1,0,0,sid1,G,T
sid2,0,1,sid2,C,C
sid3,0,2,sid3,A,G
sid4,0,3,sid4,A,C
sid5,0,4,sid5,C,G


### From VCF, via `vcf2zarr`

A `.pgen` file only ever stores hard-call genotypes, nothing else. If you need QUAL/FILTER/INFO fields, per-call
FORMAT fields (`GQ`, `DP`, `AD`, phasing, ...), or your data only exists as VCF in the first place, `read_sgkit_zarr`
reads the [sgkit](https://sgkit-dev.github.io/sgkit/) Zarr store produced by
[`vcf2zarr`](https://github.com/sgkit-dev/bio2zarr), which converts VCF files in two steps:

- `vcf2zarr explode` reads the VCF(s) once into an intermediate columnar format (ICF).
- `vcf2zarr encode` turns the ICF into the actual Zarr store; this is where the chunk sizes you'll want to think about
  are set.

Reasonable starting points: chunk sizes in the tens of thousands for variants, and large enough for samples that your
whole cohort fits in a single chunk for datasets up to a few thousand donors. Run `encode` single-process for modest
VCFs (the default); for whole-chromosome or biobank-scale conversions, use multiple worker processes together with a
memory cap (e.g. `32G`) so peak memory stays bounded.

In [4]:
from cellink.io import read_sgkit_zarr

# read_plink() and read_bgen() work the same way for PLINK1 and BGEN files
gdata_vcf = read_sgkit_zarr("../../tests/data/simulated_genotype_calls.vcz")
gdata_vcf

site-packages/sgkit/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution  # type: ignore[import]


AnnData object with n_obs × n_vars = 1000 × 1999
    var: 'PR', 'contig', 'id', 'id_mask', 'pos', 'quality', 'a0', 'a1', 'chrom'
    uns: 'n_alt', 'ploidy', 'has_phased_flag', 'has_genotype', 'has_genotype_probability', 'has_DS', 'has_mask'
    layers: None (.X), 'GENOTYPE_0', 'GENOTYPE_1', 'GENOTYPE_MASK_0', 'GENOTYPE_MASK_1', 'GENOTYPE_PHASED', 'PHASE_0'

Because it goes through the full VCF, everything the VCF carries beyond genotypes comes along too: QUAL and INFO
fields end up as extra `var` columns (here `quality` and `PR`), and any per-call FORMAT field (`GQ`, `DP`, `AD`,
phasing, ...) becomes a `layer` matching `X`'s shape. A field with one value per allele copy or allele is split into
one layer per entry, e.g. `GENOTYPE_0` and `GENOTYPE_1` for the two copies of each call, so every layer follows any
selection of donors and variants:

In [5]:
gdata_vcf.layers.keys()

KeysView(Layers with keys: None, 'GENOTYPE_0', 'GENOTYPE_1', 'GENOTYPE_MASK_0', 'GENOTYPE_MASK_1', 'GENOTYPE_PHASED', 'PHASE_0')

## Genotypes in a HiData

The genotypes become the group level `G`. The cells only need a column naming the donor of every cell, here
`donor_id`, passed as `group_id`; we simulate a few cells per donor of the PGEN file.

In [6]:
rng = np.random.default_rng(0)
donors = gdata.obs_names.to_numpy()
n_cells = 5 * len(donors)
cells = AnnData(
    X=rng.poisson(2, size=(n_cells, 20)).astype(np.float32),
    obs=pd.DataFrame(
        {"donor_id": np.repeat(donors, 5), "celltype": rng.choice(["CD4", "CD8", "NK"], size=n_cells)},
        index=[f"cell{i}" for i in range(n_cells)],
    ),
    var=pd.DataFrame(index=[f"GENE{i}" for i in range(20)]),
)

hd = cl.HiData(G=gdata, C=cells, group_id="donor_id")
hd

╔═ HiData(n_groups=100, n_cells=500) ══════════════════════════════════════════╗
║ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ ║
║ ┃ G (groups)                          ┃ C (cells, 5..5 per group)          ┃ ║
║ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ ║
║ │ AnnData object with n_obs × n_vars  │ AnnData object with n_obs × n_vars │ ║
║ │ = 100 × 1,000                       │ = 500 × 20                         │ ║
║ │     obs: 'iid', 'sex'               │     obs: 'donor_id', 'celltype'    │ ║
║ │     var: 'chrom', 'pos', 'snp_id',  │                                    │ ║
║ │ 'a0', 'a1'                          │                                    │ ║
║ │     uns: 'n_missing'                │                                    │ ║
║ └─────────────────────────────────────┴────────────────────────────────────┘ ║
╚═ group_id='donor_id' ════════════════════════════════════════════════════════╝

In [7]:
cl.validate(hd)

True

The genotypes stay on disk until they are needed. Selecting variants, donors or cells selects the matching parts of
every level, and only the selected genotypes are read when they are computed:

In [8]:
subset = hd.sel(G_var=hd.G.var["pos"] < 100, C_obs=hd.C.obs["celltype"] == "CD8")
subset.G.X

dask.array<getitem, shape=(91, 100), dtype=int8, chunksize=(50, 100), chunktype=numpy.ndarray>

In [9]:
subset.G.X.compute()[:5, :8]

array([[1, 0, 1, 2, 0, 1, 2, 0],
       [1, 0, 1, 1, 1, 1, 2, 1],
       [1, 1, 1, 0, 1, 1, 1, 1],
       [1, 0, 2, 1, 1, 1, 1, 0],
       [1, 2, 2, 2, 0, 1, 1, 2]], dtype=int8)

## Writing genotypes

{func}`cellink.io.write_zarr_hd` writes a `HiData` to a Zarr store and stores the genotypes in `G.X` as 2-bit
calls, four per byte, like `stream_pgen_to_zarr`:

- Integer values 0, 1 and 2 are stored as they are. Negative integers and NaN count as missing and stay missing.
- Imputed dosages are rounded to the nearest integer, with a warning that counts them. Pass `genotype_encoding=None`
  to keep them, or any other data in `G.X`, unchanged.
- If `G.X` holds values that do not round to 0, 1 or 2, it is written unencoded, with a warning.

`x_chunks` sets the chunk shape of `G.X` and `C.X`, together as one `(rows, columns)` tuple or separately as
`{"G": ..., "C": ...}`. A 2-bit store needs a variant chunk size divisible by 4.

In [10]:
from cellink.io import write_zarr_hd

write_zarr_hd(hd, tmp_dir / "cohort.hd.zarr", x_chunks={"G": (50, 200)})
write_zarr_hd(hd, tmp_dir / "cohort_int8.hd.zarr", x_chunks={"G": (50, 200)}, genotype_encoding=None)


def size_kb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1024


print(f"2-bit genotypes: {size_kb(tmp_dir / 'cohort.hd.zarr' / 'G' / 'X'):.0f} kB")
print(f"unencoded genotypes: {size_kb(tmp_dir / 'cohort_int8.hd.zarr' / 'G' / 'X'):.0f} kB")

2-bit genotypes: 25 kB
unencoded genotypes: 27 kB


`write_h5_hd` writes an HDF5 file instead, and `write_hd` chooses the format from the extension. The file layout
is described in the HiData [specification](https://hidata.readthedocs.io/en/latest/io/spec.html); the 2-bit
genotypes use the `cellink_pack2bit` Zarr codec, which is registered when cellink is installed. Files and stores
written with donordata or cellink 0.1.0 are read as well, and saving them again writes the HiData format.

## Reading genotypes lazily

{func}`cellink.io.read_zarr_hd` keeps dense matrices such as the genotypes on disk as Dask arrays, so selecting a
subset first only reads that subset. Missing calls are returned as NaN.

In [11]:
from cellink.io import read_zarr_hd

hd_lazy = read_zarr_hd(tmp_dir / "cohort.hd.zarr")
hd_lazy.G.X

dask.array<cellink-genotypes, shape=(100, 1000), dtype=int8, chunksize=(50, 200), chunktype=numpy.ndarray>

With `backend="zarr"`, `G.X` is a Zarr array that reads the selected donors and variants in parallel and unpacks
only those, the fastest way to subset genotypes. It returns the stored codes, with -1 for missing calls.

In [12]:
hd_fast = read_zarr_hd(tmp_dir / "cohort.hd.zarr", backend="zarr")
np.asarray(hd_fast.sel(G_obs=["S0", "S5", "S9"]).G.X)[:, :8]

array([[2, 1, 1, 2, 1, 0, 1, 1],
       [1, 2, 2, 2, 0, 1, 1, 2],
       [2, 0, 1, 1, 1, 1, 0, 1]], dtype=int8)

HDF5 files are read into memory by default; {func}`cellink.io.read_h5_hd` with `lazy=True` keeps their dense
matrices on disk as well.

In [13]:
from cellink.io import read_h5_hd, write_h5_hd

write_h5_hd(hd, tmp_dir / "cohort.hd.h5")
read_h5_hd(tmp_dir / "cohort.hd.h5", lazy=True).G.X

dask.array<array, shape=(100, 1000), dtype=int8, chunksize=(100, 1000), chunktype=numpy.ndarray>

## Next steps

With your own `HiData` in hand, {doc}`explore_annotations` covers variant QC and annotation, and
{doc}`pseudobulk_eqtl` walks through a first real analysis.